## Data Processing 


#### Loading the datasets using PyTorch Geometric's TUDataset

In [1]:
from torch_geometric.datasets import TUDataset

# loaded datasets
mutag = TUDataset(root="data/MUTAG", name="MUTAG")
proteins = TUDataset(root="data/PROTEINS", name="PROTEINS")
imdb = TUDataset(root="data/IMDB-BINARY", name="IMDB-BINARY")

# basic info
print("MUTAG:", len(mutag), "graphs")
print("PROTEINS:", len(proteins), "graphs")
print("IMDB-BINARY:", len(imdb), "graphs")

/Users/fionamagee/Robustness-of-Graph-Neural-Networks-for-Graph-Classification-Under-Structural-Perturbations/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/fionamagee/Robustness-of-Graph-Neural-Networks-for-Graph-Classification-Under-Structural-Perturbations/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Processing...
Done!
Processing...
Done!
Processing...


MUTAG: 188 graphs
PROTEINS: 1113 graphs
IMDB-BINARY: 1000 graphs


Done!


#### Checking the data - making sure graphs have valid edges, node features, and label

In [2]:
datasets = {
    "MUTAG": mutag,
    "PROTEINS": proteins,
    "IMDB-BINARY": imdb
}

for name, dataset in datasets.items():
    graph = dataset[0]

    print(f"\n{name}")
    print("Number of graphs:", len(dataset))
    print("Number of nodes:", graph.num_nodes)
    print("Number of edges:", graph.num_edges)
    print("Node features:", graph.num_node_features)
    print("Number of classes:", dataset.num_classes)
    print("Graph label:", graph.y)
    print("Edge index shape:", graph.edge_index.shape)
    print("Node feature shape:", graph.x.shape if graph.x is not None else None)


MUTAG
Number of graphs: 188
Number of nodes: 17
Number of edges: 38
Node features: 7
Number of classes: 2
Graph label: tensor([1])
Edge index shape: torch.Size([2, 38])
Node feature shape: torch.Size([17, 7])

PROTEINS
Number of graphs: 1113
Number of nodes: 42
Number of edges: 162
Node features: 3
Number of classes: 2
Graph label: tensor([0])
Edge index shape: torch.Size([2, 162])
Node feature shape: torch.Size([42, 3])

IMDB-BINARY
Number of graphs: 1000
Number of nodes: 20
Number of edges: 146
Node features: 0
Number of classes: 2
Graph label: tensor([0])
Edge index shape: torch.Size([2, 146])
Node feature shape: None


In [ ]:
for name, dataset in datasets.items():
    invalid_edges = 0
    missing_features = 0
    missing_labels = 0

    for graph in dataset:
        # checking for invalid node indices in edges
        if graph.edge_index.numel() > 0:
            if graph.edge_index.min() < 0 or graph.edge_index.max() >= graph.num_nodes:
                invalid_edges += 1

        # checking for missing node features
        if graph.x is None:
            missing_features += 1

        # checking for missing graph labels
        if graph.y is None or graph.y.numel() == 0:
            missing_labels += 1

    print(f"\n{name}")
    print("Graphs with invalid edges:", invalid_edges)
    print("Graphs with missing features:", missing_features)
    print("Graphs with missing labels:", missing_labels)


MUTAG
Graphs with invalid edges: 0
Graphs with missing features: 0
Graphs with missing labels: 0

PROTEINS
Graphs with invalid edges: 0
Graphs with missing features: 0
Graphs with missing labels: 0

IMDB-BINARY
Graphs with invalid edges: 0
Graphs with missing features: 1000
Graphs with missing labels: 0


#### Preparing node features - MUTAG and PROTEINS already have node features, but IMDB-BINARY doesn't, so you'll need to generate degree-based features.

In [8]:
from torch_geometric.transforms import BaseTransform
from torch_geometric.utils import degree

class AddDegreeFeatures(BaseTransform):
    def forward(self, data):
        data.x = degree(
            data.edge_index[0],
            num_nodes=data.num_nodes
        ).view(-1, 1)
        return data

imdb.transform = AddDegreeFeatures()

In [9]:
graph = imdb[0]

print("Number of nodes:", graph.num_nodes)
print("Node feature shape:", graph.x.shape)
print("First 10 node degrees:", graph.x[:10])

Number of nodes: 20
Node feature shape: torch.Size([20, 1])
First 10 node degrees: tensor([[ 5.],
        [ 8.],
        [19.],
        [ 6.],
        [ 5.],
        [ 5.],
        [11.],
        [ 3.],
        [ 8.],
        [ 5.]])


#### Splitting the datasets - dividing graphs into 70% training, 10% validation, and 20% testing sets.

In [10]:
from sklearn.model_selection import train_test_split
import numpy as np

def split_dataset(dataset, seed=42):
    indices = np.arange(len(dataset))

    # 1st split : 70% train, 30% remaining
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.30,
        random_state=seed,
        stratify=[dataset[i].y.item() for i in indices]
    )

    # 2nd split: 10% validation, 20% test
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=2/3,
        random_state=seed,
        stratify=[dataset[i].y.item() for i in temp_idx]
    )

    return (
        dataset[train_idx.tolist()],
        dataset[val_idx.tolist()],
        dataset[test_idx.tolist()]
    )

# Split all three datasets
mutag_train, mutag_val, mutag_test = split_dataset(mutag)
proteins_train, proteins_val, proteins_test = split_dataset(proteins)
imdb_train, imdb_val, imdb_test = split_dataset(imdb)

In [11]:
for name, train, val, test in [
    ("MUTAG", mutag_train, mutag_val, mutag_test),
    ("PROTEINS", proteins_train, proteins_val, proteins_test),
    ("IMDB-BINARY", imdb_train, imdb_val, imdb_test)
]:
    print(f"\n{name}")
    print("Training:", len(train))
    print("Validation:", len(val))
    print("Testing:", len(test))
    print("Total:", len(train) + len(val) + len(test))


MUTAG
Training: 131
Validation: 19
Testing: 38
Total: 188

PROTEINS
Training: 779
Validation: 111
Testing: 223
Total: 1113

IMDB-BINARY
Training: 700
Validation: 100
Testing: 200
Total: 1000


#### Preparing the graph structure - ensuring your perturbation code handles undirected edges correctly without introducing duplicates or self-loops

In [12]:
from torch_geometric.utils import is_undirected, contains_self_loops
from torch_geometric.utils import coalesce

for name, dataset in datasets.items():
    not_undirected = 0
    self_loops = 0
    duplicates = 0

    for graph in dataset:
        edges = graph.edge_index

        if not is_undirected(edges, num_nodes=graph.num_nodes):
            not_undirected += 1

        if contains_self_loops(edges):
            self_loops += 1

        # Check whether duplicate edges exist
        unique_edges = coalesce(edges, num_nodes=graph.num_nodes)
        if unique_edges.size(1) != edges.size(1):
            duplicates += 1

    print(f"\n{name}")
    print("Graphs that are not undirected:", not_undirected)
    print("Graphs with self-loops:", self_loops)
    print("Graphs with duplicate edges:", duplicates)


MUTAG
Graphs that are not undirected: 0
Graphs with self-loops: 0
Graphs with duplicate edges: 0

PROTEINS
Graphs that are not undirected: 0
Graphs with self-loops: 0
Graphs with duplicate edges: 0

IMDB-BINARY
Graphs that are not undirected: 0
Graphs with self-loops: 0
Graphs with duplicate edges: 0


## Algorithm 

#### Implement the GNN models

In [13]:
import torch
import torch.nn.functional as F

from torch.nn import Linear
from torch_geometric.nn import (
    GCNConv,
    SAGEConv,
    GATConv,
    global_mean_pool
)

In [14]:
class GNN(torch.nn.Module):
    def __init__(self, model_type, input_dim, num_classes, hidden_dim=64):
        super().__init__()

        self.model_type = model_type

        if model_type == "GCN":
            self.conv1 = GCNConv(input_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)

        elif model_type == "GraphSAGE":
            self.conv1 = SAGEConv(input_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        elif model_type == "GAT":
            self.conv1 = GATConv(input_dim, hidden_dim, heads=1)
            self.conv2 = GATConv(hidden_dim, hidden_dim, heads=1)

        else:
            raise ValueError("Unknown model type")

        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # First GNN layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # Second GNN layer
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Combine node embeddings into graph embeddings
        x = global_mean_pool(x, batch)

        # Predict graph classes
        x = self.classifier(x)

        return x

In [15]:
models = {}

for model_type in ["GCN", "GraphSAGE", "GAT"]:
    model = GNN(
        model_type=model_type,
        input_dim=mutag.num_node_features,
        num_classes=mutag.num_classes
    )

    models[model_type] = model

    print(f"{model_type} initialized successfully")

GCN initialized successfully
GraphSAGE initialized successfully
GAT initialized successfully


In [16]:
optimizers = {
    name: torch.optim.Adam(model.parameters(), lr=0.001)
    for name, model in models.items()
}

criterion = torch.nn.CrossEntropyLoss()

In [17]:
from torch_geometric.loader import DataLoader

loader = DataLoader(mutag_train, batch_size=32, shuffle=False)
batch = next(iter(loader))

for name, model in models.items():
    output = model(batch)

    print(f"{name}: {output.shape}")

GCN: torch.Size([32, 2])
GraphSAGE: torch.Size([32, 2])
GAT: torch.Size([32, 2])


#### Train models and establish baseline accuracy

In [18]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        output = model(batch)
        loss = criterion(output, batch.y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            output = model(batch)
            predictions = output.argmax(dim=1)

            correct += (predictions == batch.y.view(-1)).sum().item()
            total += batch.num_graphs

    return correct / total

In [19]:
import copy
import random
import numpy as np
from torch_geometric.loader import DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Organize existing dataset splits
dataset_splits = {
    "MUTAG": (mutag_train, mutag_val, mutag_test),
    "PROTEINS": (proteins_train, proteins_val, proteins_test),
    "IMDB-BINARY": (imdb_train, imdb_val, imdb_test)
}

# Store results and trained models
results = {}
trained_models = {}

epochs = 100

for dataset_name, (train_data, val_data, test_data) in dataset_splits.items():

    print(f"\n===== {dataset_name} =====")

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    for model_type in ["GCN", "GraphSAGE", "GAT"]:

        model = GNN(
            model_type=model_type,
            input_dim=train_data.num_node_features,
            num_classes=train_data.num_classes
        )

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = torch.nn.CrossEntropyLoss()

        best_val_acc = -1
        best_state = None
        best_epoch = 0

        for epoch in range(1, epochs + 1):

            loss = train(model, train_loader, optimizer, criterion)
            val_acc = evaluate(model, val_loader)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch

        # Restore best model
        model.load_state_dict(best_state)

        # Evaluate on original test graphs
        test_acc = evaluate(model, test_loader)

        # Save model and results
        trained_models[(dataset_name, model_type)] = model

        results[(dataset_name, model_type)] = {
            "validation_accuracy": best_val_acc,
            "test_accuracy": test_acc,
            "best_epoch": best_epoch
        }

        print(
            f"{model_type}: "
            f"Validation = {best_val_acc:.4f}, "
            f"Test = {test_acc:.4f}, "
            f"Best Epoch = {best_epoch}"
        )


===== MUTAG =====
GCN: Validation = 0.7895, Test = 0.6579, Best Epoch = 75
GraphSAGE: Validation = 0.8421, Test = 0.6579, Best Epoch = 52
GAT: Validation = 0.7895, Test = 0.7105, Best Epoch = 46

===== PROTEINS =====
GCN: Validation = 0.7297, Test = 0.6861, Best Epoch = 16
GraphSAGE: Validation = 0.7387, Test = 0.6861, Best Epoch = 38
GAT: Validation = 0.7387, Test = 0.6906, Best Epoch = 14

===== IMDB-BINARY =====
GCN: Validation = 0.7300, Test = 0.7100, Best Epoch = 67
GraphSAGE: Validation = 0.7100, Test = 0.7350, Best Epoch = 57
GAT: Validation = 0.6700, Test = 0.7000, Best Epoch = 60


In [21]:
import pandas as pd

baseline_results = []

for (dataset_name, model_type), metrics in results.items():
    baseline_results.append({
        "Dataset": dataset_name,
        "Model": model_type,
        "Validation Accuracy": round(metrics["validation_accuracy"] * 100, 2),
        "Test Accuracy": round(metrics["test_accuracy"] * 100, 2),
        "Best Epoch": metrics["best_epoch"]
    })

baseline_df = pd.DataFrame(baseline_results)

display(baseline_df)

,Dataset,Model,Validation Accuracy,Test Accuracy,Best Epoch
0,MUTAG,GCN,78.95,65.79,75
1,MUTAG,GraphSAGE,84.21,65.79,52
2,MUTAG,GAT,78.95,71.05,46
3,PROTEINS,GCN,72.97,68.61,16
4,PROTEINS,GraphSAGE,73.87,68.61,38
5,PROTEINS,GAT,73.87,69.06,14
6,IMDB-BINARY,GCN,73.00,71.00,67
7,IMDB-BINARY,GraphSAGE,71.00,73.50,57
8,IMDB-BINARY,GAT,67.00,70.00,60


#### Implement random structural perturbations

In [22]:
def random_edge_removal(graph, rate, seed=42):
    rng = random.Random(seed)
    modified_graph = graph.clone()

    edges = graph.edge_index.t().tolist()
    unique_edges = sorted(set(
        (min(u, v), max(u, v)) for u, v in edges
    ))

    num_remove = int(len(unique_edges) * rate)

    removed_edges = set(rng.sample(unique_edges, num_remove))

    remaining_edges = [
        (u, v) for u, v in unique_edges
        if (u, v) not in removed_edges
    ]

    new_edges = []
    for u, v in remaining_edges:
        new_edges.extend([[u, v], [v, u]])

    modified_graph.edge_index = torch.tensor(
        new_edges, dtype=torch.long
    ).t().contiguous().reshape(2, -1)

    return modified_graph

In [ ]:
def random_edge_addition(graph, rate, seed=42):
    rng = random.Random(seed)
    modified_graph = graph.clone()

    edges = graph.edge_index.t().tolist()
    unique_edges = set(
        (min(u, v), max(u, v)) for u, v in edges
    )

    num_nodes = graph.num_nodes

    num_add = int(len(unique_edges) * rate)

    possible_edges = [
        (u, v)
        for u in range(num_nodes)
        for v in range(u + 1, num_nodes)
        if (u, v) not in unique_edges
    ]

    num_add = min(num_add, len(possible_edges))

    added_edges = rng.sample(possible_edges, num_add)

    all_edges = sorted(unique_edges.union(added_edges))

    new_edges = []
    for u, v in all_edges:
        new_edges.extend([[u, v], [v, u]])

    modified_graph.edge_index = torch.tensor(
        new_edges, dtype=torch.long
    ).t().contiguous().reshape(2, -1)

    return modified_graph

In [24]:
graph = mutag_test[0]

rates = [0.05, 0.10, 0.20, 0.30]

print("Original edges:", graph.num_edges // 2)

for rate in rates:
    removed = random_edge_removal(graph, rate)
    added = random_edge_addition(graph, rate)

    print(f"\nPerturbation rate: {rate * 100:.0f}%")
    print("Edges after removal:", removed.num_edges // 2)
    print("Edges after addition:", added.num_edges // 2)

Original edges: 20

Perturbation rate: 5%
Edges after removal: 19
Edges after addition: 21

Perturbation rate: 10%
Edges after removal: 18
Edges after addition: 22

Perturbation rate: 20%
Edges after removal: 16
Edges after addition: 24

Perturbation rate: 30%
Edges after removal: 14
Edges after addition: 26


In [25]:
original = mutag_test[0]

removed = random_edge_removal(original, 0.20)
added = random_edge_addition(original, 0.20)

print("Original graph edges:", original.num_edges)
print("After removal:", removed.num_edges)
print("After addition:", added.num_edges)

print(
    "Original unchanged:",
    torch.equal(original.edge_index, mutag_test[0].edge_index)
)

Original graph edges: 40
After removal: 32
After addition: 48
Original unchanged: True


#### Implement targeted structural perturbations


In [26]:
def targeted_edge_removal(graph, rate):
    modified_graph = graph.clone()

    # Get unique undirected edges
    edges = graph.edge_index.t().tolist()
    unique_edges = sorted(set(
        (min(u, v), max(u, v)) for u, v in edges
    ))

    # Calculate node degrees
    degrees = torch.bincount(
        graph.edge_index[0],
        minlength=graph.num_nodes
    )

    # Score edges by the sum of endpoint degrees
    scored_edges = sorted(
        unique_edges,
        key=lambda edge: (
            -(degrees[edge[0]].item() + degrees[edge[1]].item()),
            edge
        )
    )

    # Remove the highest-scoring edges
    num_remove = int(len(unique_edges) * rate)
    removed_edges = set(scored_edges[:num_remove])

    remaining_edges = [
        edge for edge in unique_edges
        if edge not in removed_edges
    ]

    # Rebuild edge_index in both directions
    new_edges = []
    for u, v in remaining_edges:
        new_edges.extend([[u, v], [v, u]])

    modified_graph.edge_index = torch.tensor(
        new_edges, dtype=torch.long
    ).t().contiguous().reshape(2, -1)

    return modified_graph

In [27]:
def targeted_edge_addition(graph, rate):
    modified_graph = graph.clone()

    # Get existing undirected edges
    edges = graph.edge_index.t().tolist()
    unique_edges = set(
        (min(u, v), max(u, v)) for u, v in edges
    )

    # Calculate original node degrees
    degrees = torch.bincount(
        graph.edge_index[0],
        minlength=graph.num_nodes
    )

    # Find pairs of nodes that aren't connected
    possible_edges = [
        (u, v)
        for u in range(graph.num_nodes)
        for v in range(u + 1, graph.num_nodes)
        if (u, v) not in unique_edges
    ]

    # Prioritize pairs with the highest combined degree
    possible_edges.sort(
        key=lambda edge: (
            -(degrees[edge[0]].item() + degrees[edge[1]].item()),
            edge
        )
    )

    # Add the highest-scoring missing edges
    num_add = int(len(unique_edges) * rate)
    num_add = min(num_add, len(possible_edges))

    added_edges = possible_edges[:num_add]

    all_edges = sorted(unique_edges.union(added_edges))

    # Rebuild edge_index in both directions
    new_edges = []
    for u, v in all_edges:
        new_edges.extend([[u, v], [v, u]])

    modified_graph.edge_index = torch.tensor(
        new_edges, dtype=torch.long
    ).t().contiguous().reshape(2, -1)

    return modified_graph

In [28]:
graph = mutag_test[0]

rates = [0.05, 0.10, 0.20, 0.30]

print("Original edges:", graph.num_edges // 2)

for rate in rates:
    removed = targeted_edge_removal(graph, rate)
    added = targeted_edge_addition(graph, rate)

    print(f"\nPerturbation rate: {rate * 100:.0f}%")
    print("Edges after targeted removal:", removed.num_edges // 2)
    print("Edges after targeted addition:", added.num_edges // 2)

Original edges: 20

Perturbation rate: 5%
Edges after targeted removal: 19
Edges after targeted addition: 21

Perturbation rate: 10%
Edges after targeted removal: 18
Edges after targeted addition: 22

Perturbation rate: 20%
Edges after targeted removal: 16
Edges after targeted addition: 24

Perturbation rate: 30%
Edges after targeted removal: 14
Edges after targeted addition: 26


In [29]:
graph = mutag_test[0]

random_removed = random_edge_removal(graph, 0.20)
targeted_removed = targeted_edge_removal(graph, 0.20)

def get_unique_edges(graph):
    return set(
        (min(u, v), max(u, v))
        for u, v in graph.edge_index.t().tolist()
    )

original_edges = get_unique_edges(graph)

random_removed_edges = (
    original_edges - get_unique_edges(random_removed)
)

targeted_removed_edges = (
    original_edges - get_unique_edges(targeted_removed)
)

print("Randomly removed edges:", sorted(random_removed_edges))
print("Targeted removed edges:", sorted(targeted_removed_edges))

Randomly removed edges: [(0, 1), (2, 3), (6, 7), (7, 8)]
Targeted removed edges: [(2, 3), (3, 4), (8, 10), (13, 16)]


#### Recompute IMDB-BINARY degree features

In [30]:
def update_node_features(graph, dataset_name):
    # Only update IMDB-BINARY
    if dataset_name == "IMDB-BINARY":
        graph.x = degree(
            graph.edge_index[0],
            num_nodes=graph.num_nodes
        ).view(-1, 1)

    return graph

In [31]:
def perturb_graph(graph, dataset_name, method, rate, seed=42):

    # Make a copy to protect the original
    modified_graph = graph.clone()

    if method == "random_removal":
        modified_graph = random_edge_removal(
            modified_graph, rate, seed
        )

    elif method == "random_addition":
        modified_graph = random_edge_addition(
            modified_graph, rate, seed
        )

    elif method == "targeted_removal":
        modified_graph = targeted_edge_removal(
            modified_graph, rate
        )

    elif method == "targeted_addition":
        modified_graph = targeted_edge_addition(
            modified_graph, rate
        )

    else:
        raise ValueError("Unknown perturbation method")

    # Recalculate degree features if IMDB-BINARY
    modified_graph = update_node_features(
        modified_graph, dataset_name
    )

    return modified_graph

In [32]:
graph = imdb_test[0]

print("Original node degrees:")
print(graph.x[:10])

modified = perturb_graph(
    graph,
    dataset_name="IMDB-BINARY",
    method="targeted_removal",
    rate=0.30
)

print("\nUpdated node degrees:")
print(modified.x[:10])

print("\nFeatures changed:",
      not torch.equal(graph.x, modified.x))

print("Original graph unchanged:",
      torch.equal(graph.x, imdb_test[0].x))

Original node degrees:
tensor([[7.],
        [4.],
        [7.],
        [6.],
        [7.],
        [7.],
        [7.],
        [4.],
        [6.],
        [7.]])

Updated node degrees:
tensor([[6.],
        [3.],
        [6.],
        [5.],
        [6.],
        [6.],
        [6.],
        [3.],
        [5.],
        [6.]])

Features changed: True
Original graph unchanged: True


In [33]:
for name, dataset in [
    ("MUTAG", mutag_test),
    ("PROTEINS", proteins_test)
]:
    graph = dataset[0]

    modified = perturb_graph(
        graph,
        dataset_name=name,
        method="targeted_removal",
        rate=0.30
    )

    print(
        f"{name} features unchanged:",
        torch.equal(graph.x, modified.x)
    )

MUTAG features unchanged: True
PROTEINS features unchanged: True


#### Verify the perturbation methods

In [34]:
from torch_geometric.utils import (
    is_undirected,
    contains_self_loops,
    coalesce,
    degree
)

def verify_perturbation(original, modified, dataset_name, method, rate):

    # Count unique undirected edges
    original_edges = original.num_edges // 2
    modified_edges = modified.num_edges // 2

    requested = int(original_edges * rate)

    # Number of edges added or removed
    if "removal" in method:
        actual = original_edges - modified_edges
        expected = requested
    else:
        actual = modified_edges - original_edges

        # Account for graphs with insufficient available edges
        max_possible = original.num_nodes * (original.num_nodes - 1) // 2
        available = max_possible - original_edges
        expected = min(requested, available)

    edge_count_valid = actual == expected

    # Check undirected structure
    undirected = is_undirected(
        modified.edge_index,
        num_nodes=modified.num_nodes
    )

    # Check for self-loops
    no_self_loops = not contains_self_loops(
        modified.edge_index
    )

    # Check for duplicate edges
    unique_edges = coalesce(
        modified.edge_index,
        num_nodes=modified.num_nodes
    )

    no_duplicates = (
        unique_edges.size(1) == modified.edge_index.size(1)
    )

    # Check node features
    if dataset_name == "IMDB-BINARY":
        expected_features = degree(
            modified.edge_index[0],
            num_nodes=modified.num_nodes
        ).view(-1, 1)

        features_valid = torch.equal(
            modified.x, expected_features
        )
    else:
        features_valid = torch.equal(
            original.x, modified.x
        )

    return {
        "edge_count": edge_count_valid,
        "undirected": undirected,
        "no_self_loops": no_self_loops,
        "no_duplicates": no_duplicates,
        "features": features_valid
    }

In [35]:
methods = [
    "random_removal",
    "random_addition",
    "targeted_removal",
    "targeted_addition"
]

rates = [0.05, 0.10, 0.20, 0.30]

test_datasets = {
    "MUTAG": mutag_test,
    "PROTEINS": proteins_test,
    "IMDB-BINARY": imdb_test
}

verification_results = []

for dataset_name, dataset in test_datasets.items():

    for method in methods:
        for rate in rates:

            failures = 0

            for i, graph in enumerate(dataset):

                modified = perturb_graph(
                    graph,
                    dataset_name=dataset_name,
                    method=method,
                    rate=rate,
                    seed=42 + i
                )

                checks = verify_perturbation(
                    graph, modified, dataset_name, method, rate
                )

                if not all(checks.values()):
                    failures += 1

            verification_results.append({
                "Dataset": dataset_name,
                "Method": method,
                "Rate": f"{int(rate * 100)}%",
                "Graphs Tested": len(dataset),
                "Failures": failures,
                "Status": "PASS" if failures == 0 else "FAIL"
            })

In [36]:
import pandas as pd

verification_df = pd.DataFrame(verification_results)

display(verification_df)

print("\nTotal configurations:", len(verification_df))
print("Total failures:", verification_df["Failures"].sum())

if verification_df["Failures"].sum() == 0:
    print("All perturbation methods passed verification!")
else:
    print("Some checks failed. Further inspection needed.")

,Dataset,Method,Rate,Graphs Tested,Failures,Status
0,MUTAG,random_removal,5%,38,0,PASS
1,MUTAG,random_removal,10%,38,0,PASS
2,MUTAG,random_removal,20%,38,0,PASS
3,MUTAG,random_removal,30%,38,0,PASS
4,MUTAG,random_addition,5%,38,0,PASS
5,MUTAG,random_addition,10%,38,0,PASS
6,MUTAG,random_addition,20%,38,0,PASS
7,MUTAG,random_addition,30%,38,0,PASS
8,MUTAG,targeted_removal,5%,38,0,PASS
9,MUTAG,targeted_removal,10%,38,0,PASS



Total configurations: 48
Total failures: 0
All perturbation methods passed verification!
